# Daten bereinigen und analysieren

In der Informatik ist es häufig notwendig, Daten zu bereinigen, bevor sie weiterverarbeitet oder analysiert werden können, weil zum Beispiel Sensoren fehlerhafte Werte liefern oder Daten unvollständig sind.

**Merke:** Bereinigen bedeutet fehlerhafte, unvollständige oder widersprüchliche Daten zu erkennen und durch sinnvolle Ersatzwerte oder Korrekturen zu verbessern – damit die anschließende Analyse zuverlässige Ergebnisse liefert.

> **Verfälschungen vermeiden:** Beim Bereinigen von Daten darfst du keine persönlichen Annahmen oder Wertungen einbringen. Verwende nur nachvollziehbare, objektive Methoden wie den Mittelwert oder benachbarte Messwerte. Die ursprüngliche Bedeutung der Daten soll erhalten bleiben – sonst kann die Analyse verfälscht werden.

Lade dir `messwerte.csv` von der Kursseite herunter und speichere sie im selben Ordner wie dieses Notebook, bevor du die Zellen ausführst.

## 1. CSV-Datei einlesen

Mit dem `csv`-Modul kannst du eine Tabelle mit Messwerten zeilenweise einlesen.

In [ ]:
import csv  # Modul zum Arbeiten mit CSV-Dateien

def einlesen_datei(pfad):
    daten = []
    with open(pfad, newline='', encoding='utf-8') as datei:
        reader = csv.DictReader(datei)  # jede Zeile wird als Wörterbuch eingelesen
        for zeile in reader:
            daten.append(zeile)
    return daten

messwerte = einlesen_datei("messwerte.csv")
print("Anzahl der Datensätze:", len(messwerte))

## 2. Fehlende Werte löschen

Wenn Temperaturwerte fehlen (z. B. `"N/A"` oder leer), entfernen wir die ganze Zeile.

In [ ]:
def entferne_fehlende(daten, spalte):
    neue_daten = []
    geloescht = 0  # Zähler für gelöschte Zeilen
    for eintrag in daten:
        wert = eintrag[spalte]
        if wert != "N/A" and wert.strip() != "":
            neue_daten.append(eintrag)
        else:
            geloescht += 1  # eine ungültige Zeile wurde übersprungen
    print(f"{geloescht} Zeilen wurden wegen fehlender Werte in '{spalte}' entfernt.")
    return neue_daten

messwerte = entferne_fehlende(messwerte, "Temperatur")

## 3. Unrealistische Werte ersetzen

Werte unter −30 °C oder über 50 °C gelten als unrealistisch. Wir ersetzen sie durch den Mittelwert aus Nachbarwerten.

In [ ]:
def ersetze_unrealistische(messwerte, spalte, min_wert, max_wert):
    for i in range(len(messwerte)):
        wert = float(messwerte[i][spalte])
        if wert < min_wert or wert > max_wert:
            if i > 0 and i < len(messwerte) - 1:
                vorher = float(messwerte[i - 1][spalte])
                nachher = float(messwerte[i + 1][spalte])
                ersatz = (vorher + nachher) / 2
                messwerte[i][spalte] = ersatz  # Wert durch Mittelwert der Nachbarn ersetzen

ersetze_unrealistische(messwerte, "Temperatur", -30, 50)
print("Unrealistische Werte ersetzt.")

## 4. Statistik berechnen

Für jeden Tag berechnen wir: Durchschnitt, Minimum und Maximum der Temperatur.

In [ ]:
def temperatur_statistik(daten):
    tage = {}
    for eintrag in daten:
        tag = eintrag["Zeitpunkt"].split(" ")[0]  # Datum extrahieren
        temperatur = float(eintrag["Temperatur"])
        if tag not in tage:
            tage[tag] = []
        tage[tag].append(temperatur)

    for tag in tage:
        werte = tage[tag]
        durchschnitt = sum(werte) / len(werte)
        print(f"{tag}: Mittel = {durchschnitt:.1f}°C, Min = {min(werte):.1f}°C, Max = {max(werte):.1f}°C")

temperatur_statistik(messwerte)

### Aufgabe

Falls du es noch nicht getan hast: Lies `messwerte.csv` ein, entferne Zeilen mit fehlenden Temperaturwerten, ersetze unrealistische Werte (unter -30 °C oder über 50 °C), und gib die Temperaturstatistik pro Tag auf der Konsole aus. Nutze dafür die Zellen oben, oder schreibe den ganzen Ablauf einmal selbstständig in die Zelle unten.

In [ ]:
# Eigene Lösung: kompletten Ablauf selbstständig schreiben
# 1. messwerte.csv einlesen
# 2. fehlende Werte entfernen
# 3. unrealistische Werte ersetzen
# 4. Temperaturstatistik pro Tag ausgeben


## 5. Reguläre Ausdrücke (Wiederholung)

Reguläre Ausdrücke (kurz Regex) kennst du bereits aus Klassenstufe 10 (Sprachen): Es handelt sich um Suchmuster, mit denen du Text nach bestimmten Regeln durchsuchen, überprüfen oder ersetzen kannst – zum Beispiel um zu prüfen, ob eine Eingabe einem bestimmten Format entspricht. In Python steht dafür das Modul `re` zur Verfügung.

**Merke:** Ein regulärer Ausdruck beschreibt ein Muster aus normalen Zeichen und Sonderzeichen (z. B. `\d` für eine Ziffer, `+` für "ein- oder mehrmals"), mit dem geprüft werden kann, ob ein Text diesem Muster entspricht.

Auf dieser Seite verwendete Sonderzeichen:

| Zeichen | Bedeutung |
|---|---|
| `^` | Anfang des Textes |
| `$` | Ende des Textes |
| `\d` | eine Ziffer (0-9) |
| `\w` | ein Wortzeichen (Buchstabe, Ziffer oder Unterstrich) |
| `+` | das vorangehende Zeichen ein- oder mehrmals |
| `{n}` | das vorangehende Zeichen genau n-mal |
| `[...]` | eines der Zeichen innerhalb der eckigen Klammern |
| `\.` | ein tatsächlicher Punkt |

> **Hinweis zu `^` und `$`:** Bei `re.fullmatch()` muss ohnehin der komplette Text passen – `^`/`$` machen aber deutlich, dass keine zusätzlichen Zeichen davor oder danach erlaubt sein sollen. Bei `re.search()` ohne `^`/`$` reicht es, wenn das Muster irgendwo im Text vorkommt.

In [ ]:
import re

muster = r"^\d{5}$"  # genau 5 Ziffern
text = "01067"

if re.fullmatch(muster, text):
    print("Gültige Postleitzahl")
else:
    print("Ungültige Postleitzahl")

### Aufgabe: Erste einfache Prüfungen

Schreibe für jedes der folgenden Alltagsbeispiele eine `re.search()`-Abfrage, die prüft, ob ein bestimmtes Zeichen oder Muster in einem Text vorkommt:

- a) Enthält ein Text das Zeichen `@` (z. B. um grob zu prüfen, ob es sich um eine E-Mail-Adresse handeln könnte)?
- b) Enthält ein Satz ein Ausrufezeichen `!` oder ein Fragezeichen `?`?
- c) Beginnt ein Name mit einem Großbuchstaben?
- d) Besteht eine Eingabe nur aus Ziffern (z. B. eine Hausnummer ohne Zusatz)?

In [ ]:
import re

# a) Enthält text das Zeichen @?


# b) Enthält satz ein ! oder ?


# c) Beginnt name mit einem Großbuchstaben?


# d) Besteht eingabe nur aus Ziffern?


### Aufgabe: Alltagsmuster mit regulären Ausdrücken prüfen

Entwickle reguläre Ausdrücke für die folgenden vier Alltagsbeispiele und teste sie mit `re.fullmatch()`:

- a) Eine Uhrzeit im Format HH:MM (z. B. 14:30)
- b) Ein Datum im Format TT.MM.JJJJ (z. B. 24.12.2025)
- c) Eine einfache E-Mail-Adresse (etwas@etwas.etwas)
- d) Ein Produktcode für Verkaufsautomaten: genau drei Zeichen – `G` (Getränk) oder `S` (Snack), dann eine Ziffer von 1 bis 9, dann eine Ziffer von 0 bis 9 (gültig: `G10`, `S31`; ungültig: `G01`, `X12`, `S3`). Dieses Format brauchst du auf Seite 3.6 wieder.

In [ ]:
import re

# a) Uhrzeit HH:MM


# b) Datum TT.MM.JJJJ


# c) einfache E-Mail-Adresse


# d) SnackBot-Produktcode


---
## Lösungen

Schau erst hier rein, nachdem du es selbst probiert hast!

In [ ]:
import re

text = "Kontakt: a.mueller@schule.de"
satz = "Wirklich?!"
name = "Anna"
eingabe = "12345"

print(bool(re.search(r"@", text)))          # a)
print(bool(re.search(r"[!?]", satz)))       # b)
print(bool(re.match(r"^[A-Z]", name)))      # c)
print(bool(re.fullmatch(r"\d+", eingabe)))  # d)

In [ ]:
import re

print(bool(re.fullmatch(r"^\d{2}:\d{2}$", "14:30")))                       # a)
print(bool(re.fullmatch(r"^\d{2}\.\d{2}\.\d{4}$", "24.12.2025")))          # b)
print(bool(re.fullmatch(r"^[\w.]+@[\w.]+\.\w+$", "etwas@etwas.etwas")))    # c)
print(bool(re.fullmatch(r"^[GS][1-9]\d$", "G10")))                          # d)